# Lezione 6 — Outlier Detection avanzata

Notebook di laboratorio per una lezione di circa **5 ore** su:
- **outlier puntuali**
- **outlier contestuali** nelle **serie temporali**
- **outlier multivariati e collettivi**

## Obiettivi didattici
Alla fine del laboratorio gli studenti dovrebbero saper:
1. distinguere tra outlier puntuali, contestuali e collettivi
2. applicare metodi semplici e metodi ML-based
3. capire quando la rappresentazione del problema è più importante dell'algoritmo
4. leggere e commentare i risultati in modo critico

In [1]:
# Import di base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

np.random.seed(42)
pd.set_option("display.max_columns", 50)

## 0) Richiamo teorico rapido

### Outlier puntuali
Sono **singole osservazioni** molto lontane dal comportamento normale del dataset.

### Outlier contestuali
Sono osservazioni anomale **rispetto al contesto**.
Nelle serie temporali il contesto può essere:
- il momento della giornata
- la stagione
- il trend
- il livello atteso in quel periodo

### Outlier collettivi
Sono **gruppi di osservazioni** che, prese singolarmente, possono sembrare normali,
ma nel loro insieme descrivono un comportamento anomalo.

# 1) Outlier puntuali

Iniziamo dal caso più semplice: pochi punti anomali in un dataset 2D.
Useremo:
- distanza euclidea dal centro
- MAD / modified z-score
- Isolation Forest

In [2]:
# TODO: genera il dataset sintetico per gli outlier puntuali
# 1. crea i punti normali con make_blobs
# 2. genera alcuni outlier lontani
# 3. concatena tutto
# 4. costruisci un DataFrame con x1, x2 e is_anomaly


In [3]:
# TODO 2: ricrea il grafico di questa sezione e commentalo in 2-3 righe.


## 1.1 Metodo semplice: distanza euclidea dal centro

Idea:
1. stimiamo un centro dei dati
2. misuriamo la distanza di ogni punto da quel centro
3. i punti molto lontani vengono considerati anomali

In [4]:
# TODO: calcola la distanza euclidea dal centro
# 1. stima il centro globale
# 2. calcola la distanza di ogni punto dal centro
# 3. scegli una soglia
# 4. crea pred_euclid


## 1.2 MAD — Median Absolute Deviation

Il MAD è una misura **robusta** di dispersione:
\[
MAD = median(|x - median(x)|)
\]

Lo applichiamo a ciascuna feature e segnaliamo come anomalo
un punto che ha almeno una coordinata con `|modified_z| > 3.5`.

In [5]:
# TODO: implementa il MAD e il modified z-score
# 1. definisci una funzione mad(arr)
# 2. calcola il modified z-score per x1 e x2
# 3. definisci pred_mad


## 1.3 Isolation Forest

Isolation Forest isola i punti anomali più facilmente dei punti normali.

In [6]:
# TODO: applica Isolation Forest agli outlier puntuali
# 1. scala le feature
# 2. istanzia il modello
# 3. usa fit_predict
# 4. converti l'output in 0/1


In [7]:
# TODO 6: completa questa cella.


# 2) Outlier contestuali nelle serie temporali

## Elementi tipici di una serie temporale
- **trend**: andamento di lungo periodo
- **stagionalità**: pattern che si ripete
- **rumore**: fluttuazione casuale
- **anomalie**: osservazioni che si discostano da quanto atteso

Un valore può essere “normale” in assoluto ma anomalo rispetto a:
- il periodo dell'anno
- l'ora del giorno
- la fase del ciclo
- il trend atteso

In [8]:
# TODO: costruisci una serie temporale con trend, stagionalità e rumore
# 1. crea il tempo t
# 2. definisci trend, season e noise
# 3. genera la serie
# 4. inserisci anomalie contestuali
# 5. crea ts


In [9]:
# TODO 8: ricrea il grafico di questa sezione e commentalo in 2-3 righe.


## 2.1 Baseline ingenua: z-score globale

In [10]:
# TODO: calcola una baseline con z-score globale
# 1. calcola media e deviazione standard globali
# 2. costruisci z_global
# 3. definisci pred_global
# 4. valuta il risultato


## 2.2 STL decomposition

**STL** = Seasonal-Trend decomposition using Loess

Separiamo la serie in:
- trend
- stagionalità
- residuo

Le anomalie contestuali dovrebbero emergere nei **residui**.

> Se `statsmodels` non fosse disponibile: `pip install statsmodels`

In [11]:
# TODO: applica STL decomposition
# 1. importa STL
# 2. decomponi la serie con il periodo corretto
# 3. salva trend, stagionalità e residui


In [12]:
# TODO 11: ricrea il grafico di questa sezione e commentalo in 2-3 righe.


In [13]:
# TODO: rileva anomalie sui residui STL
# 1. calcola mediana e MAD dei residui
# 2. costruisci un modified z-score
# 3. definisci pred_stl
# 4. valuta il risultato


## 2.3 Prophet

**Prophet** è un modello additivo per serie temporali che rappresenta:
- trend
- stagionalità
- eventi / holiday (se presenti)

Nel nostro caso lo usiamo per modellare il comportamento atteso
e poi cercare anomalie nei residui.

> Se Prophet non fosse installato: `pip install prophet`

In [14]:
# TODO 13: completa questa cella.


In [15]:
# TODO 14: completa questa cella.


In [16]:
# TODO: usa Prophet per modellare la serie attesa
# 1. prepara il DataFrame con ds e y
# 2. importa Prophet
# 3. addestra il modello
# 4. ottieni yhat e i residui
# 5. definisci pred_prophet


# 3) Outlier multivariati e collettivi

## 3.1 Correlazione anomala tra misure
Due variabili possono essere normalmente molto correlate.
Un punto che rompe questa relazione può essere anomalo anche se i singoli valori non sono estremi.

In [17]:
# TODO: costruisci un dataset con misure correlate
# 1. genera m1 e m2 con relazione lineare + rumore
# 2. aggiungi punti che rompano la correlazione
# 3. crea corr_df


In [18]:
# TODO 17: ricrea il grafico di questa sezione e commentalo in 2-3 righe.


### Idea semplice: residui rispetto a una relazione lineare

Stimiamo una relazione `m2 ~ m1` e guardiamo i residui.
Punti con residui molto grandi possono essere anomalie.

In [19]:
# TODO: rileva anomalie che rompono la correlazione
# 1. stima la retta m2 ~ m1
# 2. calcola m2_hat e i residui
# 3. usa un criterio robusto sui residui
# 4. definisci pred_resid


## 3.2 Outlier collettivi: segmenti anomali in una serie

Costruiamo una serie con stagionalità normale e inseriamo un segmento
con comportamento anomalo. Poi trasformiamo la serie in **finestre mobili**.

In [20]:
# TODO: costruisci una serie con un segmento anomalo
# 1. genera una serie normale
# 2. sostituisci un intervallo con un pattern anomalo
# 3. crea ts2 e marca il segmento


In [21]:
# TODO: crea finestre mobili e feature di finestra
# 1. scorri la serie con una finestra mobile
# 2. calcola mean, std, range e slope
# 3. costruisci win_df


In [22]:
# TODO: applica Isolation Forest alle finestre
# 1. scala le feature di finestra
# 2. addestra Isolation Forest
# 3. crea pred_collective_window
# 4. valuta il risultato


# 4) Sintesi finale

## Cosa abbiamo visto

### Outlier puntuali
- distanza euclidea
- MAD / modified z-score
- Isolation Forest

### Outlier contestuali
- una serie temporale non va letta solo “globalmente”
- STL e Prophet permettono di modellare trend e stagionalità
- l’anomalia emerge spesso nei residui

### Outlier multivariati / collettivi
- una relazione tra variabili può rompersi
- un segmento può essere anomalo nel suo insieme
- spesso bisogna cambiare rappresentazione (es. finestre mobili)

## Messaggio chiave
Non basta chiedersi:
> “Quale algoritmo uso?”

Bisogna chiedersi prima:
> “Che cosa significa, in questo problema, essere anomali?”